# 01 — Exploração dos Dados CVM

Notebook exploratório para análise dos dados fundamentalistas baixados da CVM.

**Pré-requisito:** Execute o pipeline antes de rodar este notebook:
```bash
cd valuation_cvm
python -m src.main --start-year 2019 --end-year 2025
```

In [ ]:
import sys
sys.path.insert(0, '..')  # Adiciona o diretório raiz do projeto

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('Ambiente configurado.')

## 1. Cadastro de Empresas

In [ ]:
from src.company_mapper import load_company_registry, filter_company_by_name_or_cvm

cadastro = load_company_registry()
print(f'Total de empresas no cadastro: {len(cadastro)}')
cadastro.head()

In [ ]:
# Buscar Petrobras
petrobras = filter_company_by_name_or_cvm('PETROBRAS')
print('Petrobras encontrada:')
petrobras[['CD_CVM', 'CNPJ_CIA', 'DENOM_CIA', 'SIT']]

In [ ]:
# Buscar Vale
vale = filter_company_by_name_or_cvm('VALE')
vale[['CD_CVM', 'CNPJ_CIA', 'DENOM_CIA', 'SIT']]

In [ ]:
# Buscar Itaú
itau = filter_company_by_name_or_cvm('ITAU')
itau[['CD_CVM', 'CNPJ_CIA', 'DENOM_CIA', 'SIT']].head(10)

## 2. DRE — Demonstração de Resultado

In [ ]:
from src.financial_statements import load_processed_statement, extract_account

dre = load_processed_statement('DRE', 'DFP')
print(f'DRE DFP: {len(dre):,} registros')
dre.dtypes

In [ ]:
# Obter CD_CVM da Petrobras
cd_cvm_petrobras = petrobras.iloc[0]['CD_CVM'] if not petrobras.empty else None
print(f'CD_CVM Petrobras: {cd_cvm_petrobras}')

# Extrair receita líquida da Petrobras
if cd_cvm_petrobras:
    receita = extract_account(dre, cd_cvm=cd_cvm_petrobras, account_keywords=['receita líquida', 'receita de venda'])
    print(f'\nReceita Líquida — {len(receita)} registros encontrados')
    if not receita.empty:
        cols = ['DT_REFER', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA_AJUSTADO', 'ESCALA_MOEDA']
        cols_ok = [c for c in cols if c in receita.columns]
        print(receita[cols_ok].sort_values('DT_REFER', ascending=False).head(10))

## 3. BPA — Balanço Patrimonial Ativo

In [ ]:
bpa = load_processed_statement('BPA', 'DFP')
print(f'BPA DFP: {len(bpa):,} registros')

if cd_cvm_petrobras:
    caixa = extract_account(bpa, cd_cvm=cd_cvm_petrobras, account_keywords=['caixa e equivalentes'])
    print(f'\nCaixa e Equivalentes — {len(caixa)} registros')
    if not caixa.empty:
        cols = ['DT_REFER', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA_AJUSTADO']
        cols_ok = [c for c in cols if c in caixa.columns]
        print(caixa[cols_ok].sort_values('DT_REFER', ascending=False).head(5))

## 4. BPP — Balanço Patrimonial Passivo

In [ ]:
bpp = load_processed_statement('BPP', 'DFP')
print(f'BPP DFP: {len(bpp):,} registros')

if cd_cvm_petrobras:
    # Dívida bruta
    divida = extract_account(bpp, cd_cvm=cd_cvm_petrobras, account_keywords=['empréstimos', 'financiamentos'])
    print(f'\nEmpréstimos e Financiamentos — {len(divida)} registros')
    if not divida.empty:
        cols = ['DT_REFER', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA_AJUSTADO']
        cols_ok = [c for c in cols if c in divida.columns]
        print(divida[cols_ok].sort_values('DT_REFER', ascending=False).head(10))

## 5. Snapshot Financeiro Completo

In [ ]:
from src.financial_statements import build_company_snapshot
from src.valuation_metrics import calculate_basic_metrics

if cd_cvm_petrobras:
    snap = build_company_snapshot(cd_cvm_petrobras)
    metrics = calculate_basic_metrics(snap)

    print('=== SNAPSHOT FINANCEIRO — PETROBRAS ===')
    for k, v in metrics.items():
        if not k.startswith('has_'):
            if isinstance(v, float) and not np.isnan(v) and abs(v) > 1000:
                print(f'  {k:<35}: R$ {v:>20,.0f}')
            elif isinstance(v, float):
                print(f'  {k:<35}: {v:>10.4f}')
            else:
                print(f'  {k:<35}: {v}')

## 6. Cálculo de Métricas Básicas

In [ ]:
if cd_cvm_petrobras:
    print('Margem Bruta:   ', f"{metrics.get('margem_bruta', float('nan'))*100:.1f}%" if metrics.get('margem_bruta') else 'N/D')
    print('Margem EBIT:    ', f"{metrics.get('margem_ebit', float('nan'))*100:.1f}%" if metrics.get('margem_ebit') else 'N/D')
    print('Margem Líquida: ', f"{metrics.get('margem_liquida', float('nan'))*100:.1f}%" if metrics.get('margem_liquida') else 'N/D')
    print('ROE:            ', f"{metrics.get('roe', float('nan'))*100:.1f}%" if metrics.get('roe') else 'N/D')
    print('ROA:            ', f"{metrics.get('roa', float('nan'))*100:.1f}%" if metrics.get('roa') else 'N/D')

## 7. Cálculo Preliminar de EPV

In [ ]:
from src.valuation_epv import epv_from_ebit_series

# NOTA: Substitua pelos valores reais extraídos da DRE da empresa
# Este é apenas um exemplo com dados hipotéticos
ebit_historico_exemplo = [10_000_000_000, 12_000_000_000, 8_000_000_000, 15_000_000_000, 11_000_000_000]
divida_liq_exemplo = 50_000_000_000

epv = epv_from_ebit_series(
    ebit_series=ebit_historico_exemplo,
    tax_rate=0.34,        # IRPJ + CSLL Brasil
    wacc=0.12,            # Premissa de WACC (ajuste conforme análise)
    net_debt=divida_liq_exemplo,
    norm_method='median',
)

print('=== EPV ENTERPRISE ===')
print(f"EBIT Normalizado (mediana): R$ {epv['ebit_normalized']:,.0f}")
print(f"NOPAT:                      R$ {epv['nopat']:,.0f}")
print(f"EPV Enterprise Value:       R$ {epv['epv_enterprise']:,.0f}")
print(f"Dívida Líquida:             R$ {epv['net_debt']:,.0f}")
print(f"EPV Equity:                 R$ {epv['epv_equity']:,.0f}")
print(f"Flags: {epv['flags']}")

## 8. Cálculo Preliminar de DCF

In [ ]:
from src.valuation_dcf import calculate_dcf

# NOTA: Substitua pelos valores reais extraídos do DFC da empresa
dcf = calculate_dcf(
    base_fcf=8_000_000_000,
    growth_rates=[0.08, 0.08, 0.06, 0.06, 0.05],  # 5 anos de projeção
    terminal_growth=0.03,
    wacc=0.12,
    net_debt=50_000_000_000,
)

print('=== DCF ===')
print(f"Enterprise Value:    R$ {dcf['enterprise_value']:,.0f}")
print(f"Equity Value:        R$ {dcf['equity_value']:,.0f}")
print(f"VP do Valor Terminal: R$ {dcf['vp_valor_terminal']:,.0f}")
print(f"% do EV no TV: {dcf['premissas'].get('pct_ev_de_valor_terminal', 'N/D')}%")
print(f"Flags: {dcf['flags']}")

## 9. Ticker Mapper

O arquivo `data/processed/ticker_mapper.csv` contém o mapeamento entre CD_CVM e ticker.
Preencha a coluna TICKER manualmente ou via outra API (ex.: brapi.dev).

In [ ]:
from pathlib import Path

ticker_path = Path('../data/processed/ticker_mapper.csv')
if ticker_path.exists():
    mapper = pd.read_csv(ticker_path, dtype=str)
    print(f'Ticker mapper: {len(mapper)} empresas')
    # Mostrar apenas as que têm ticker preenchido
    com_ticker = mapper[mapper['TICKER'].notna() & (mapper['TICKER'] != '')]
    print(f'Com ticker preenchido: {len(com_ticker)}')
    mapper.head(10)
else:
    print('ticker_mapper.csv não encontrado. Execute o pipeline primeiro.')